## Data Mining Project (DATDRD05-HC-T06):  Predicting Customer Churn, Revenue at Risk, and Customer Segmentation for a B2B company 

## Table of Contents

1. Introduction  
2. Dataset Description  
3. Data Preparation and Feature Engineering
4. Defining the Target Variable: Customer Churn
5. Constructing the Final Modelling Dataset
6. Machine Learning Models
   6.1a Preparing Data for Machine Learning
   6.1b Feature Standardisation
   6.2 Splitting the Dataset into Training and Testing Sets
   6.3 k-Nearest Neighbors (kNN) Model
   6.4 Naive Bayes Model
   6.5 Logistic Regression

7. Model Evaluation & Comparison
8. Customer Risk Segmentation (Extension)
9. Revenue at Risk Analysis
10. Revenue at Risk by Customer Risk Segment
11. Predicted Churn
    11.1 Distribution of Predicted Churn
    11.2 Predicted Churn Rate
    11.3 High-Value Customers at Risk

12. Customer Risk Segmentation
13. Estimating Short-Term Customer Lifetime Value (CLV) 
14. Conclusion 
15. Deployment: Interactive Churn Prediction Tool


# 1. Introduction

## Project Objective

The objective of this project is to use transactional Enterprise Resource Planning (ERP) data from a B2B company, in order to generate actionable business insights that support decision-making within the company with their customer base being a focus area. Data Mining principles and techniques taught during the course DATDRD05-HC-T06 are employed. Since, its a B2B company that relies heavily on its repeat customers, a key business challenge is identifying customers who are likely to stop purchasing so that the company may take proactive actions to retain its customers and reduce potential losses.  

## Understanding: Why this is important?
Conceptually-speaking, 'customer churn' is the process whereby a customer stops purchasing products from a company.
Customer churn thus would have a direct impact on revenue, and thus comprehending the financial rammifications is also pertinant. If a company can identify customers who are likely to stop purchasing, it can take proactive measures such as targeted marketing or account management interventions for which segmentation 

This makes churn prediction not only a technical problem, but also a critical business problem.
More specifically, based on the dynamics of this business problem, it can be broken down into four key objectives: 

1. **Customer Churn Prediction**  
   Predicting whether a customer is likely to terminate purchasing from the company 

2. **Revenue at Risk Estimation**  
   Estimate potential revenue losses from customers predicted to churn to support other functions of decision-making within the company, such as the Finance and Budgeting functions. 

3. **Customer Risk Segmentation**  
   Classificaion of customers into risk categories for better understanding 

4. **Short-Term Customer Lifetime Value (CLV)**  
   Estimating the short-term value of customers 


## How will this be done?
To achieve these objectives, three machine learning models are developed and compared:

- k-Nearest Neighbors (kNN)  
- Naive Bayes  
- Logistic Regression
  
These models will learn patterns from the customer behaviour information in the datasets, and use them to predict whether a customer is likely to churn. Each model is then evaluated based on predictive performance, following which revenue at risk, customer segmentation and short-term CLV is calculated. 

### What will be the outcome?

The outcome of this project is a predictive system that:

1. Identifies customers at risk of churn
2. Quantifies potential revenue loss for the company
3. Supports data-driven decision making for further steps 

Thus, it enables the company to move from 'reactive to proactive' customer engagement and management.

## 2. Dataset Description

### Which data is selected for usage?

The dataset consists of transactional B2B sales data extracted directly from the ERP system of Nordic Industrial Supply B.V., over a period of three years: 2023–2025. 

The three datasets used are:

1. **Customer_Adres_AccMng.xlsx**  
  Contains customer details such as customer ID (Debiteur), location, account manager, etc. 

2. **Customer_Turnover per month.xlsx**  
  Contains the monthly revenue per each customer.

3. **Order intake per day_Amount and Turnover.xlsx**  
  Contains detailed daily transactional data 

### Why are multiple datasets needed and what does allow for?

Each dataset captures a different aspect of customer behaviour that is relevant for analysis:

1. Customer dataset --> Who the customer is  
2. Turnover dataset --> How much did the customer spend  
3. Order dataset --> How often did the customer purchase  

By combining these datasets, a more comprehensive picture of customer behaviour can be constructed satisfying the project objective. 
This combination enables the foundation for predicting customer churn by:

--> Analysis of purchasing patterns over time  
--> Identification of behavioural trends  
--> Creation of features that can be used for machine learning  



In [48]:
import pandas as pd
import numpy as np

customers = pd.read_excel("Customer_Adres_AccMng.xlsx")
monthly_turnover = pd.read_excel("Customer_Turnover per month.xlsx")
orders = pd.read_excel("Order intake per day_Amount and Turnover.xlsx")

customers.head()
customers.columns

monthly_turnover.head()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,Debiteur,Jaar,Periode,Omzet
0,8000,2024,4,172.80
1,8000,2024,10,232.35
2,8000,2024,11,0.00
3,8000,2025,2,0.00
4,13001,2023,1,6542.50


## 3. Data Preparation and Feature Engineering

The turnover dataset records revenue on a monthly level, wherein each row represents a customer’s activity within a specific month.
However, machine learning models require a dataset wherein each row represents only a single observation.
In this case, the observation would be the customer.

### Why is this necessary?

Since models such as kNN and Naive Bayes cannot work directly with transactional data wherein one customer appears multiple times, each customer must be represented by a single row containing information about their behaviour in summary form. 

### How will this be done?

The transactional data at this stage is first transformed into a customer-level dataset by aggregating revenue per customer and generating behavioural features.

Thus, two key features will be created:

1. Total Revenue: The total value in revenue generated by the customer
2. Active Months: The number of months that a customer purchases for

It must be noted that these features allow us to summarize both the customer value and their respective activity level.

In [49]:
customer_revenue = monthly_turnover.groupby("Debiteur").agg({
    "Omzet": "sum",
    "Periode": "count"
}).reset_index()

customer_revenue.columns = ["Debiteur", "total_revenue", "active_months"]

customer_revenue.head()

,Debiteur,total_revenue,active_months
0,8000,405.15,4
1,13001,90238.25,9
2,13002,5101.36,1
3,13006,492014.52,28
4,13008,165164.85,21


### Additional Feature: Average Monthly Revenue

An additional feature has to be created in order to enhance the ability to capture average spending behaviour of a customer.

### Why is this important?

Total revenue alone does not reflect how consistently a customer spends; two customers may have the same total revenue but very different purchasing patterns in terms of bheaviour and/or period of time. 

### How is it calculated?

Average Monthly Revenue = Total Revenue / Active Months

In [50]:
customer_revenue["avg_monthly_revenue"] = (
    customer_revenue["total_revenue"] / customer_revenue["active_months"]
)

customer_revenue.head()

,Debiteur,total_revenue,active_months,avg_monthly_revenue
0,8000,405.15,4,101.287500
1,13001,90238.25,9,10026.472222
2,13002,5101.36,1,5101.360000
3,13006,492014.52,28,17571.947143
4,13008,165164.85,21,7864.992857


### Interpretation of Average Monthly Revenue

This feature provides insight into what can be described as 'purchasing intensity'. This can be important for churn prediction, as changes in a set purchasing behaviour over a period of time can potentially signal that a customer may stop buying.




## 4. Defining the Target Variable: Customer Churn

### What is the next step?

In order to train a supervised machine learning model such as one employed in this project, a target variable must be defined. In this case, the goal is to predict customer churn and will thus be the target variable. 
'Customer churn' as a target variable refers to the situation wherein the customer stops purchasing products from the company.

### Why is this necessary: What is 'target variable'? 

Machine learning models require a known outcome or 'target variable', in order to learn patterns. However, since this dataset does not explicitly contain a churn label, it must be derived from customer behaviour as being key to predict churn. 

### How is churn defined?

A common approach in churn analysis is to define churn based on inactivity; customers that have not made purchases for a certain prolonged period are likely to have discontinued their relationship with the company.

Thus, in this project, churn is defined based on the recency of customer transactions with the company per customer. 

### Step 1: Identifying the last purchase

The first step would be to determine the most recent transaction for each customer from the dataset. 

In [51]:
last_purchase = monthly_turnover.groupby("Debiteur").agg({
    "Jaar": "max",
    "Periode": "max"
}).reset_index()

last_purchase.head()

,Debiteur,Jaar,Periode
0,8000,2025,11
1,13001,2025,12
2,13002,2023,2
3,13006,2025,12
4,13008,2025,12


### Interpretation of most recent Purchase Data

The output above shows, for each customer:

1. Jaar: The most recent year
2. Periode: The most recent month

This is used to identify when each customer last made their last purchase which forms the basis for measuring customer inactivity.



### Step 2: Converting Dates into a Continuous Time Scale

### What is being done? 

The dataset stores time using separate variables for year and month. To calculate inactivity, these have to be combined into a single measure of time.

### Why is this necessary?

In order to determine how long it has been since a customer’s most recent or last purchase, a continuous time scale allows calculation of the difference between time periods.

In [52]:
last_purchase["month_index"] = last_purchase["Jaar"] * 12 + last_purchase["Periode"]

last_purchase.head()

,Debiteur,Jaar,Periode,month_index
0,8000,2025,11,24311
1,13001,2025,12,24312
2,13002,2023,2,24278
3,13006,2025,12,24312
4,13008,2025,12,24312


### Interpretation of Month Index

The new variable "month_index" represents time as a continuous sequence of months, which allows for comparison of time periods in numerical order, which is essential for being able to calculate true customer inactivity.

### Step 3: Calculating Customer Inactivity

### What is being done? 

The months that have passed since each customer’s last purchase has to be calculated. 

### Why is this important?

Customer inactivity is one of the strongest indicators of churn. Customers who have not purchased for a long time are more likely to have stopped or potentially stop buying completely.

### Interpretation of Inactivity

The variable "months_since_last_purchase" shows how long each customer has been inactive for.

- Low values (e.g., 0–2 months) → active customers  
- High values (e.g., 12+ months) → potentially churned customers  

This provides a clear behavioural signal for identifying churn.

### Step 4: Defining the Churn Indicator

### What are we doing?

We convert inactivity into a binary churn variable.

### Why is it necessary?

Machine learning models require a target variable that is evident and clear. A binary variable allows the model to classify customers as churned or active.

### How is churn defined?

A customer can be classified as churned if:

- They have been inactive for **12 months or more**

This threshold reflects a full year of inactivity, which in turn presents a strong indicator of churn.

In [11]:
last_purchase["churn"] = (
    last_purchase["months_since_last_purchase"] >= 12
).astype(int)

last_purchase.head()

,Debiteur,Jaar,Periode,month_index,months_since_last_purchase,churn
0,8000,2025,11,24311,1,0
1,13001,2025,12,24312,0,0
2,13002,2023,2,24278,34,1
3,13006,2025,12,24312,0,0
4,13008,2025,12,24312,0,0


### Interpretation of the Churn Variable

The 'churn' variable thus represents the target variable as:

- 1 --> being a Customer that has churned  
- 0 --> being that the Customer is still active  

From the output it can be noted that:

- Customers with high inactivity are labeled as 'churned'  
- Customers with recent activity are labeled as active  

This variable will be later be used to train machine learning models to predict churn.

## 4.1 Customer Risk Segmentation Based on Inactivity

### What are we doing?

In addition to defining churn as a binary variable, customers are further categorized into levels of risk, based on their inactivity.

### Why is this important?

A binary churn variable - 0 or 1 in this case- provides for a clear classification, but it does have the ability to not capture different levels of risk. In practice, businesses have the potential to benefit from understanding **degrees of risk**, rather than a simple yes/no outcome.
This could also allow for more targeted and efficient decision-making.

### How are the levels of risk in this case defined?

Customers are segmented into three categories based on their inactivity, in order to provide a more nuanced view of customer behaviour for understanding: 

- **Low Risk** → Recently active customers  
- **Medium Risk** → Customers with moderate inactivity  
- **High Risk** → Customers with long inactivity  



In [12]:
def risk_category(months):
    if months >= 12:
        return "High Risk"
    elif months >= 6:
        return "Medium Risk"
    else:
        return "Low Risk"

last_purchase["risk_category"] = last_purchase["months_since_last_purchase"].apply(risk_category)

last_purchase.head()

,Debiteur,Jaar,Periode,month_index,months_since_last_purchase,churn,risk_category
0,8000,2025,11,24311,1,0,Low Risk
1,13001,2025,12,24312,0,0,Low Risk
2,13002,2023,2,24278,34,1,High Risk
3,13006,2025,12,24312,0,0,Low Risk
4,13008,2025,12,24312,0,0,Low Risk


### Interpretation of Level of Risk Categories

The newly variable "risk_category" provides a rather comprehensive overview of customer behaviour:

- **Low Risk** --> Customers that have purchased recently and are thus, likely still engaged  
- **Medium Risk** --> Customers that show signs of reduced activity and thus, may require attention  
- **High Risk** --> Customers that have been inactive for a longer period of time, and thus and are likely to churn  

This segmentation is created to enhance the practical usefulness of this analysis by allowing the company to prioritize actions based on not just customer churn, but risk level.
For example:

- High-risk customers can be targeted with targetted retention strategies  
- Medium-risk customers can be monitored and engaged occassionally 
- Low-risk customers can be maintained through standard relationship management tools in place 

This step adds a business-oriented layer to the analysis, moving predictions towards actionable insights for the company. 

## 5. Constructing the Final Modelling Dataset

## What happens at this stage? 

At this stage, the analysis has now produced two key components for analysis:

1. Customer behavioural features, derived from aggregated transaction data  
2. The churn indicator, derived from customer inactivity  

These components must now be combined into a single dataset.

### Why is this necessary?

As mentioned previously, machine learning models require a dataset wherein:

- Each row represents a single observation (customer)  
- Each column represents either a feature or the target variable  

Combining these elements is necessary to ensure that the model can learn the relationship between customer behaviour and churn.

### How does this work? 

The datasets are merged using the customer identifier (**Debiteur**) as the linking key, to ensure that each customer’s behavioural features are aligned with their respective and corresponding churn label. 

### What will be the outcome? 

The resulting dataset forms the final modelling dataset, which will be then used to train and evaluate the proposed machine learning models.

In [13]:
model_data = customer_revenue.merge(
    last_purchase[["Debiteur", "churn"]],
    on="Debiteur",
    how="left"
)

model_data.head()

,Debiteur,total_revenue,active_months,avg_monthly_revenue,churn
0,8000,405.15,4,101.287500,0
1,13001,90238.25,9,10026.472222,0
2,13002,5101.36,1,5101.360000,1
3,13006,492014.52,28,17571.947143,0
4,13008,165164.85,21,7864.992857,0


### Interpretation of Final Dataset

The above dataset forms the foundation for predictive modelling, as it contains:

- Behavioural features (total_revenue, active_months, avg_monthly_revenue)  
- The target variable (churn)  

Each row thereby represents a single customer, making the dataset in its current state suitable for machine learning.
From the output, customers with different behavioural patterns are associated with either churned or active labels. This allows the model to learn which behaviours can be later linked to churn. 



## 6. Machine Learning Models

### 6.1a Preparing Data for Machine Learning

The dataset will divided into input features (X) and a target variable (y).

### Why is this necessary?

This is because machine learning models require the following:

- Input features --> variables used to make predictions  
- Target variable --> the outcome to be predicted  

Separating these allows the model to learn patterns between in this case, customer behaviour and churn.

### Features that are used: 

The explanatory variables represent customer behaviour:

1. Total Revenue --> The overall monetary value in revenue   
2. Active Months --> The frequency of purchasing by customer 
3. Average Monthly Revenue --> The spending intensity  

### Target variables wherein: 

- 1 = churned customer  
- 0 = active customer  

In [15]:
X = model_data[["total_revenue", "active_months", "avg_monthly_revenue"]]
y = model_data["churn"]

X.head()

,total_revenue,active_months,avg_monthly_revenue
0,405.15,4,101.287500
1,90238.25,9,10026.472222
2,5101.36,1,5101.360000
3,492014.52,28,17571.947143
4,165164.85,21,7864.992857


--> Can be influenced by feature magnitude  

Standardisation ensures fair contribution of all features, thereby improving model performance and reliability. After scaling:

- All features are comparable  
- No single variable could dominate the model  



In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 6.2 Splitting the Dataset into Training and Testing Sets

In order to evaluate the performance of a machine learning model, the dataset must be split into training and testing data. 

### Why is this necessary?

This is necessary in order to evaluate model performance on unseen data and prevent overfitting. Here: 

- Training data --> is used to learn patterns  
- Testing data --> is used to evaluate performance  
    
### How is it done?

The dataset is split by using a standard approach in machine learning which is :

- 80% for training  
- 20% for testing  


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((450, 3), (113, 3))

### Interpretation of Scaled Data

The scaled datasets are now prepared for machine learning.
It can be noted for further interpretation that all features are now standardized and comparable, and : 

- X_train_scaled --> used for training  
- X_test_scaled --> used for evaluation  

### Additional Note: Why is scaling applied after splitting?

Scaling is performed after splitting the dataset so that data leakage can be prevented. If scaling were applied before the splitting step, information from the test set could be influenced by the training process, which would then lead to an overly optimistic model performance.

By fitting the scaler only on the training data and applying it to the test data, evaluation fair and unbiased in the process. 


## 6.3 k-Nearest Neighbors (kNN) Model

The first machine learning model applied in this analysis is k-Nearest Neighbors (kNN) model. 

### What is the kNN model?

kNN is a supervised classification algorithm that has the ability to predict the class of an observation, based on the majority class of its nearest neighbours in a certain feature space.

### How does it work in this context?

For customer churn prediction, the model classifies a customer as either: 
- Churned (1)  
- Active (0)  

This is done by comparing each customer to others that have similar behavioural characteristics:

- Total revenue generated  
- Number of active purchasing months  
- Average monthly revenue  

The assumption would be that customers with similar behaviours tend to exhibit similar outcomes or results for the company, such as churn in this case. 

### Why is kNN suitable here?

kNN is a choice of model in this project due to multiple relevant reasons: 
1. It captures similarity between customers  
2. It does not assume a certain fixed relationship between the variables  
3. It works suitably well with behavioural data, which is what is available in this case 

### Additional: Why has k = 5 been chosen?

The parameter k here represents the number of neighbours that are considered when making predictions.
A small k may lead to overfitting, whereas a large k might oversimplify patterns. Thus, a value of k = 5 strikes a balance between sensitivity and stability for this case. 


In [27]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train model
knn_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)

y_pred_knn[:10]

array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])

### Interpretation of kNN Predictions

The model predicts whether each customer in the test dataset is likely to churn, based on similarities in customer behaviour: 

- 1 = predicted to churn  
- 0 = predicted to remain active  



### Model Evaluation: k-Nearest Neighbors

The model is evaluated using the testing dataset, which contains unseen data.
Evaluation ensures that the model is able to generalize to new customers and is not simply memorizing the training data.

### Metrics used:

- Accuracy → overall correctness  
- Confusion Matrix → types of prediction errors  
- Classification Report → precision, recall, and F1-score  

In [28]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy_knn = accuracy_score(y_test, y_pred_knn)
accuracy_knn

0.7699115044247787

### Interpretation of kNN Model Performance

The kNN model achieved an accuracy of approximately 77%. 

However, accuracy alone is not sufficient, especially for churn prediction.

In [29]:
conf_matrix_knn = confusion_matrix(y_test, y_pred_knn)
conf_matrix_knn

print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.82      0.80      0.81        70
           1       0.69      0.72      0.70        43

    accuracy                           0.77       113
   macro avg       0.76      0.76      0.76       113
weighted avg       0.77      0.77      0.77       113



### Interpretation of kNN Model Performance

The k-Nearest Neighbors model achieved an accuracy of approximately 77%, indicating a solid predictive performance.

The classification report shows:

- Precision (churn): 0.69  
- Recall (churn): 0.72  
- F1-score: 0.70  

In conclusion, this means that the model correctly identifies around 72% of customers who actually churn.

---

### Business Insight: 

While the model performs reasonably well in detecting churned customers, some factors are still missed (false negatives).
From a business perspective:

- Missing churned customers may lead to lost revenue  
- However, the model still provides useful signals for identifying at-risk customers  

Overall, still the kNN provides a strong baseline for churn prediction.

### 6.4 Naive Bayes Model

Naive Bayes is a probabilistic classification algorithm based on Bayes' Theorem.

### What does it assume?

It assumes that all input features are independent.



In [30]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

nb_model.fit(X_train_scaled, y_train)
y_pred_nb = nb_model.predict(X_test_scaled)

accuracy_nb = accuracy_score(y_test, y_pred_nb)
accuracy_nb

confusion_matrix(y_test, y_pred_nb)
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.95      0.57      0.71        70
           1       0.58      0.95      0.72        43

    accuracy                           0.72       113
   macro avg       0.76      0.76      0.72       113
weighted avg       0.81      0.72      0.72       113



### Interpretation of Naive Bayes

The Naive Bayes model achieved an accuracy of approximately 72%.

The classification report shows:

- High recall for churn (0.95) → most churned customers are detected  
- Lower precision (0.58) → many false positives  

### What does this mean?

The model tends to predict churn too often. It rarely misses churned customers, but could potentially incorrectly flag active customers as churn. 

### Business Insight: 

This model may be useful when the goal is to avoid missing churned customers. However, the high number of false positives could result in inefficient use of resources, as well as many customers being targeted unnecessarily which could have a negative affect as well. 

### 6.5 Logistic Regression Model

Logistic Regression estimates the probability of churn.

### Why would it be useful?

- Interpretable and Widely used  
- Provides insight into feature relationships which could be important in this case 

In [32]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()

log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

accuracy_log = accuracy_score(y_test, y_pred_log)

confusion_matrix(y_test, y_pred_log)
print(classification_report(y_test, y_pred_log))

              precision    recall  f1-score   support

           0       0.87      0.76      0.81        70
           1       0.67      0.81      0.74        43

    accuracy                           0.78       113
   macro avg       0.77      0.79      0.77       113
weighted avg       0.79      0.78      0.78       113



### Interpretation of Logistic Regression

The Logistic Regression model achieved an accuracy of approximately 78%, the highest among the tested models.

The classification report shows:

- Recall (churn): 0.81  
- Precision (churn): 0.67  

This means that the model identifies most churned customers while maintaining a reasonable precision for the project objective. 

### Business Insight: 

Logistic Regression provides a strong balance between:

- Identifying churned customers  
- Avoiding excessive false positives  

## 7. Model Evaluation & Comparison

### Model Performance:

- kNN → 0.77  
- Naive Bayes → 0.72  
- Logistic Regression → 0.78  

### Interpretation

- Logistic Regression model achieves the highest accuracy and provides balanced performance.

- kNN performs slightly lower but remains effective.

- Naive Bayes detects most churned customers but also produces many false positives.



Note: It is important to note that model performance can vary based both data splits and preprocessing, highlighting the importance of reproducibility and evaluation.

## 8. Customer Risk Segmentation (Extension)

### What is added beyond basic modelling?

Instead of only predicting churn (0 or 1), customers are segmented based on their probability of churn.

### Why is this important?

Businesses need to understand different levels of risk in order to:

- Prioritize high-risk customers  
- Allocate resources efficiently  
- Design targeted retention strategies  

### How is this implemented?

Using the kNN model, predicted probabilities are used to classify customers into:

- High Risk → immediate action required  
- Medium Risk → monitoring required  
- Low Risk → stable customers  

In [34]:
# Get churn probabilities
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

# Define risk categories
def risk_level(prob):
    if prob > 0.7:
        return "High Risk"
    elif prob > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

risk_categories = [risk_level(p) for p in y_prob_knn]

# Create dataframe
risk_df = pd.DataFrame({
    "Actual": y_test.values,
    "Probability": y_prob_knn,
    "Risk_Level": risk_categories
})

risk_df.head()

,Actual,Probability,Risk_Level
0,0,0.6,Medium Risk
1,1,0.8,High Risk
2,0,0.0,Low Risk
3,0,0.0,Low Risk
4,1,0.6,Medium Risk


### Interpretation of Risk Segmentation

This approach provides a more detailed understanding of customer churn risk.

--> High-risk customers are most likely to churn and thus would require immediate intervention  
--> Medium-risk customers could be monitored  
--> Low-risk customers are relatively stable  

### Business Value

This transforms the model into a decision-support tool. Instead of simply predicting churn, the company can now use this to prioritize retention efforts, optimize marketing resources, and reduce revenue loss effectively. 

## 9. Revenue at Risk Analysis

### What is Revenue at Risk?

Revenue at Risk represents the potential revenue loss associated with customers who are predicted to churn.

### Why is this important?

This is also important because while predicting churn is useful, businesses ultimately need to understand the impact in:

- How much revenue is at risk  
- Which customers contribute most to potential losses 

### How is it calculated?

Revenue at risk is estimated by:

1. Identifying customers predicted to churn  
2. Linking those customers to their revenue contribution  
3. Summing the revenue of at-risk customers  


In [35]:
# Add predictions to dataset
model_data["predicted_churn"] = knn_model.predict(
    scaler.transform(X)
)

# Calculate revenue at risk
revenue_at_risk = model_data.loc[
    model_data["predicted_churn"] == 1, "total_revenue"
].sum()

revenue_at_risk

np.float64(2082702.4700000002)

### Interpretation of Revenue at Risk

The calculated revenue at risk represents the total revenue generated by customers who are predicted to churn.
This value supports and gives a direct estimate of the potential financial loss if these customers are not retained and continue to churn.
This is also important for comprehensive analysis, for instance losing a small number of high-value customers may have a greater impact than losing many low-value customers.

## 10. Revenue at Risk by Customer Risk Segment

Instead of treating all churn predictions equally, revenue at risk can be broken down by risk level. This could provide insights to the company for: 
- Which segments contribute most to potential revenue loss  
- Where intervention should be prioritized  

In [36]:
# Combine risk segmentation with revenue
risk_df["total_revenue"] = model_data.loc[
    y_test.index, "total_revenue"
].values

# Revenue by risk level
revenue_by_risk = risk_df.groupby("Risk_Level")["total_revenue"].sum()

revenue_by_risk

Risk_Level
High Risk       127632.02
Low Risk       6714760.37
Medium Risk     174058.43
Name: total_revenue, dtype: float64

### Interpretation of Revenue by Risk Segment

This breakdown shows how revenue is distributed across different risk levels.

- High-risk customers represent the most immediate threat  
- Medium-risk customers provide opportunities for early intervention  
- Low-risk customers contribute stable revenue  

---

### Business Insight: 

This allows the company to:

- Focus retention efforts on high-risk, high-value customers  
- Design targeted strategies for different segments  
- Maximize return on retention investment  

This demonstrates how machine learning outputs can be translated into actionable business insights.

## 11. Predicted Churn

This section analyzes how many customers are predicted to churn versus remain active.

In [42]:
model_data["predicted_churn"].value_counts()

predicted_churn
0    348
1    215
Name: count, dtype: int64

### 11.1 Distribution of Predicted Churn

The output shows the number of customers predicted to churn compared to those predicted to remain active.

This provides a basic understanding of how the model classifies the customer base.


### 11.2 Predicted Churn Rate

The predicted churn rate represents the proportion of customers expected to churn.
This metric helps assess overall customer stability and urgency for retention strategies.

In [43]:
churn_rate = model_data["predicted_churn"].mean()
churn_rate

np.float64(0.38188277087033745)

### 11.3 High-Value Customers at Risk 

This identifies customers who are both high-value and at risk of churn.

These customers should be prioritized, as losing them would result in significant financial impact.

In [44]:
high_value_churn = model_data[
    model_data["predicted_churn"] == 1
].sort_values("total_revenue", ascending=False)

high_value_churn.head(10)

,Debiteur,total_revenue,active_months,avg_monthly_revenue,churn,predicted_churn
512,15384,95700.32,4,23925.080000,1,1
285,14094,78661.50,3,26220.500000,1,1
474,15260,62462.05,9,6940.227778,1,1
509,15379,58492.50,6,9748.750000,1,1
439,15140,47329.40,2,23664.700000,1,1
312,14219,46585.00,2,23292.500000,1,1
239,13960,45741.00,6,7623.500000,1,1
335,14273,42808.50,5,8561.700000,1,1
347,14309,40537.34,5,8107.468000,1,1
462,15231,39594.00,4,9898.500000,0,1


## 12. Customer Risk Segmentation

Customers are segmented based on both churn prediction and revenue contribution.

In [45]:
model_data["risk_segment"] = "Low"

median_revenue = model_data["total_revenue"].median()

model_data.loc[
    (model_data["predicted_churn"] == 1) & (model_data["total_revenue"] >= median_revenue),
    "risk_segment"
] = "High Value - High Risk"

model_data.loc[
    (model_data["predicted_churn"] == 1) & (model_data["total_revenue"] < median_revenue),
    "risk_segment"
] = "Low Value - High Risk"

model_data.loc[
    (model_data["predicted_churn"] == 0) & (model_data["total_revenue"] >= median_revenue),
    "risk_segment"
] = "High Value - Low Risk"

model_data.loc[
    (model_data["predicted_churn"] == 0) & (model_data["total_revenue"] < median_revenue),
    "risk_segment"
] = "Low Value - Low Risk"

model_data["risk_segment"].value_counts()

risk_segment
High Value - Low Risk     262
Low Value - High Risk     195
Low Value - Low Risk       86
High Value - High Risk     20
Name: count, dtype: int64

This segmentation groups customers into:

- High Value – High Risk  
- High Value – Low Risk  
- Low Value – High Risk  
- Low Value – Low Risk  

---

### Business Value

This allows the company to prioritize retention efforts based on both risk and financial importance.

## 13. Estimating Short-Term Customer Lifetime Value (CLV) at Risk

In [46]:
model_data["estimated_clv_risk"] = (
    model_data["avg_monthly_revenue"] * 12 * model_data["predicted_churn"]
)

model_data[model_data["predicted_churn"] == 1].sort_values(
    "estimated_clv_risk", ascending=False
).head(10)

,Debiteur,total_revenue,active_months,avg_monthly_revenue,churn,predicted_churn,risk_segment,estimated_clv_risk
460,15223,33240.00,1,33240.000,1,1,High Value - High Risk,398880.00
285,14094,78661.50,3,26220.500,1,1,High Value - High Risk,314646.00
512,15384,95700.32,4,23925.080,1,1,High Value - High Risk,287100.96
439,15140,47329.40,2,23664.700,1,1,High Value - High Risk,283976.40
312,14219,46585.00,2,23292.500,1,1,High Value - High Risk,279510.00
542,15426,28684.00,2,14342.000,0,1,High Value - High Risk,172104.00
551,15436,23853.57,2,11926.785,0,1,High Value - High Risk,143121.42
19,13048,23200.00,2,11600.000,1,1,Low Value - High Risk,139200.00
86,13260,11040.00,1,11040.000,0,1,Low Value - High Risk,132480.00
453,15188,10954.00,1,10954.000,1,1,Low Value - High Risk,131448.00


This estimates potential future revenue loss over a one-year period.
---
### Business Insight

This helps identify customers whose churn would result in the greatest financial impact, and not just immediate losses in revenue. 

## 14. Conclusion

This project applied machine learning techniques to predict customer churn using transactional ERP data.
Three models were evaluated:

- k-Nearest Neighbors (kNN)
- Naive Bayes
- Logistic Regression

Logistic Regression achieved the highest accuracy, while kNN provided strong behavioural insights.

---

### Key Findings: 

- Customer activity is the strongest predictor of churn  
- Revenue alone does not prevent churn  
- A significant portion of revenue is at risk  

---

### Business Impact: 

The analysis enables:

- Identification of at-risk customers  
- Estimation of financial impact  
- Targeted retention strategies  

---

### Final Insight: 

This project demonstrates how data mining can be used to transform raw transactional data into actionable business decisions.

## 15. Deployment: Interactive Churn Prediction Tool

In final stage of the project involves deploying the predictive model as an interactive web application using Streamlit.
The application allows a user to input customer-level features available: total revenue, number of active months, and average monthly revenue. Based on these inputs, the trained model predicts whether a customer is likely to churn, along with the associated probability and respective risk level.
This deployment transforms the model from a purely analytical tool into a practical decision-support system. It enables users in the business to interact with the model in real time and assess customer risk without requiring the technical expertise.
Such an approach enhances the practical applicability of data mining techniques and demonstrates how predictive models can be integrated into business processes to support customer retention strategies.